# Deck Census and Similarity Exploration

## Goal

Build a reproducible census of exact 60-card decks and explore deck similarity.
Validation-deck selection is handled separately by `select_validation_decks.ipynb`.


## Setup

The only paths to adjust are visible below. Exact deck IDs and archetypes are computed by reusable functions in `analysis.deck_statistics`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd() / 'imitation_learning', Path.cwd().parent]
    for candidate in candidates:
        if (candidate / 'analysis' / 'deck_statistics.py').is_file():
            return candidate.resolve()
    raise FileNotFoundError('Could not locate the imitation_learning project root')


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.deck_statistics import (
    CardCatalog,
    build_deck_census,
    build_similarity_pairs,
    load_deck_rows,
)

DECK_DATA_DIR = PROJECT_ROOT / 'data' / 'deck'
CARD_TABLE_PATH = PROJECT_ROOT.parent / 'pokemon_tcg_ai_battle' / 'EN_Card_Data.csv'
OUTPUT_DIR = PROJECT_ROOT / 'data' / 'deck_analysis'

print('Deck data:', DECK_DATA_DIR)
print('Card table:', CARD_TABLE_PATH)
print('Output:', OUTPUT_DIR)


## Load & Validate

Every successfully extracted episode must have exactly two rows, players 0 and 1, and each row must contain a valid 60-card list.

In [ ]:
deck_paths = sorted(DECK_DATA_DIR.glob('*.decks.csv'))
if not deck_paths:
    raise FileNotFoundError(f'No .decks.csv files found under {DECK_DATA_DIR}')

deck_facts = load_deck_rows(deck_paths)
catalog = CardCatalog.from_csv(CARD_TABLE_PATH)

print('Files:', len(deck_paths))
print('Deck rows:', len(deck_facts))
print('Episodes:', deck_facts[['date', 'episode_id']].drop_duplicates().shape[0])
print('Dates:', ', '.join(sorted(deck_facts['date'].unique())))

fact_preview = deck_facts.drop(columns=['deck']).head(10)
display(fact_preview)

## Deck Census

A row represents one exact sorted 60-card multiset. Rank follows usage, while `deck_id` remains stable across reruns.

In [ ]:
deck_summary = build_deck_census(deck_facts, catalog)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / 'deck_summary.csv'
deck_summary.to_csv(summary_path, index=False, encoding='utf-8-sig')

print('Unique exact decks:', len(deck_summary))
print('Saved:', summary_path)
display(
    deck_summary[
        ['rank', 'deck_archetype', 'deck_id', 'uses', 'usage_percent', 'wins', 'losses', 'draws', 'win_rate', 'unique_opponent_decks']
    ].head(100)
)

## Pairwise Similarity

`changed_slots` is the minimum number of card slots that must be replaced. Weighted Jaccard is count-aware and ranges from 0 to 1.

In [ ]:
similarity_pairs = build_similarity_pairs(deck_summary)
similarity_path = OUTPUT_DIR / 'deck_similarity_pairs.csv'
similarity_pairs.to_csv(similarity_path, index=False, encoding='utf-8-sig')

print('Unordered deck pairs:', len(similarity_pairs))
print('Saved:', similarity_path)
display(similarity_pairs.head(30))

## Checks

These assertions make silent changes in extraction or distance definitions visible.

In [ ]:
expected_pairs = len(deck_summary) * (len(deck_summary) - 1) // 2
assert abs(deck_summary['usage_percent'].sum() - 100.0) < 1e-8
assert (
    deck_summary['wins'] + deck_summary['losses'] + deck_summary['draws']
).equals(deck_summary['uses'])
assert len(similarity_pairs) == expected_pairs
if not similarity_pairs.empty:
    assert similarity_pairs['changed_slots'].between(0, 60).all()
    assert similarity_pairs['weighted_jaccard'].between(0, 1).all()

print('All census and similarity checks passed.')